In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
import torchaudio
import zipfile
import time
import math
import ast
import os

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

from sklearn.model_selection import train_test_split

In [ ]:
drive.mount('/content/drive')

zip_path = "/content/drive/MyDrive/dataset.zip"
extract_path = "/content/dataset/"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

model_save_path = "/content/drive/MyDrive/SSL_Projesi/CRNN/"

test_csv = '/content/drive/MyDrive/SSL_Projesi/test_metadata.csv'
test_df = pd.read_csv(test_csv)

train_csv = '/content/drive/MyDrive/SSL_Projesi/train_metadata.csv'
train_df = pd.read_csv(train_csv)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
class STFTDataset(Dataset):
    def __init__(self, df, n_fft=1024, hop_length=512, preload=True):
        self.df = df.reset_index(drop=True)
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.preload = preload

        self.window = torch.hann_window(self.n_fft)

        self.stfts = []
        self.targets = []
        self.angles = []

        if self.preload:
            self._preload_data()

    def _preload_data(self):
        print(f"{len(self.df)} dosya STFT + IPD hazırlanıyor...", flush=True)

        for _, row in self.df.iterrows():
            # 1. Ses dosyasını yükle (Hata buradaydı, düzeltildi)
            wav, _ = torchaudio.load(row["audio_path"])

            # 2. STFT Hesaplama
            stft_complex = torch.stft(
                wav,
                n_fft=self.n_fft,
                hop_length=self.hop_length,
                window=self.window,
                return_complex=True
            )

            # 3. Real/Imaginary ayırma ve şekillendirme
            stft_ri = torch.view_as_real(stft_complex)
            stft_ri = stft_ri.permute(0, 3, 1, 2)
            stft_ri = stft_ri.reshape(-1, stft_ri.shape[2], stft_ri.shape[3])

            # 4. Faz ve IPD (Yön) hesaplamaları
            phase = torch.angle(stft_complex)
            ref = phase[7:8]  # center mic

            ipd = phase[:7] - ref
            ipd_sin = torch.sin(ipd)
            ipd_cos = torch.cos(ipd)
            ipd = torch.cat([ipd_sin, ipd_cos], dim=0)

            # 5. Normalizasyon ve Birleştirme
            stft_ri = (stft_ri - stft_ri.mean()) / (stft_ri.std() + 1e-6)
            stft_final = torch.cat([stft_ri, ipd], dim=0)

            # 6. Hedef (Açı/Azimuth) Hesaplama (Sin/Cos dönüşümü)
            az = float(row["azimuth_deg"])
            rad = math.radians(az)
            target = torch.tensor(
                [math.sin(rad), math.cos(rad)],
                dtype=torch.float32
            )

            # 7. Listelere ekleme (targets buraya düzgünce eklendi)
            self.stfts.append(stft_final)
            self.targets.append(target)
            self.angles.append(torch.tensor(az, dtype=torch.float32))

        print("Preload tamamlandı.", flush=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return self.stfts[idx], self.targets[idx], self.angles[idx]

In [ ]:
nw = min(2, os.cpu_count() or 1)

train_df_split, val_df_split = train_test_split(
    train_df,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

train_dataset = STFTDataset(train_df_split, preload=True)
val_dataset = STFTDataset(val_df_split, preload=True)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True,
    num_workers=nw,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=nw,
    pin_memory=True,
    drop_last=False
)

In [ ]:
#LEakyRELU

class CRNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(30, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 1)),
            nn.Dropout(0.2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 1)),
            nn.Dropout(0.2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 1)),
            nn.Dropout(0.2),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, 30, 513, 32)
            out = self.features(dummy)

            _, c, h, _ = out.shape
            gru_input_size = c * h

        self.gru = nn.GRU(
            input_size=gru_input_size,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.2
        )

        self.attention = nn.Sequential(
            nn.Linear(256, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        self.head = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 2) # Çıktı: [sin_pred, cos_pred]
        )

    def forward(self, x):
        features = self.features(x)

        features = features.permute(0, 3, 1, 2)
        batch_size, time_frames, _, _ = features.shape
        features = features.reshape(batch_size, time_frames, -1)

        gru_out, _ = self.gru(features)

        attn = self.attention(gru_out)
        attn = torch.softmax(attn, dim=1)

        pooled = torch.mean(gru_out * attn, dim=1)

        out = self.head(pooled)
        out = F.normalize(out, dim=-1)

        return out

In [ ]:
def angular_error_deg(pred_sin_cos: torch.Tensor,
                      true_angles_deg: torch.Tensor) -> float:
    pred_deg = torch.rad2deg(
        torch.atan2(pred_sin_cos[:, 0], pred_sin_cos[:, 1])
    )
    pred_deg = (pred_deg + 360.0) % 360.0
    diff = torch.abs(pred_deg - true_angles_deg)
    return torch.min(diff, 360.0 - diff).mean().item()

In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, delta=0.1, path='best_ssl_model.pth'):
        self.patience  = patience
        self.delta     = delta
        self.path      = path
        self.counter   = 0
        self.best_err  = float('inf')
        self.early_stop = False

    def __call__(self, val_error_deg: float, model: nn.Module):
        if val_error_deg < self.best_err - self.delta:
            self.best_err = val_error_deg
            torch.save(model.state_dict(), self.path)
            self.counter = 0
        else:
            self.counter += 1
            print(f"    [EarlyStopping] Sayaç: {self.counter}/{self.patience} "
                  f"(en iyi: {self.best_err:.2f}°)")
            if self.counter >= self.patience:
                self.early_stop = True

In [ ]:
NUM_EPOCHS = 100

history = []

model = CRNN().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
criterion = torch.nn.MSELoss()

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=4
)

current_path = f"best_model.pth"
current_path = os.path.join(model_save_path, current_path)

early_stopping = EarlyStopping(patience=15, delta=0.01, path=current_path)

total_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):

    ep_start = time.time()

    # ---------------- TRAIN ----------------
    model.train()
    train_loss, train_err = 0, 0

    for wav, targets, angle in train_loader:
        wav, targets, angle = wav.to(device), targets.to(device), angle.to(device)

        optimizer.zero_grad()

        pred = model(wav)
        loss = criterion(pred, targets)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)

        optimizer.step()

        train_loss += loss.item()
        train_err += angular_error_deg(pred, angle)

    avg_train_err = train_err / len(train_loader)
    avg_train_loss = train_loss / len(train_loader)

    # ---------------- VAL ----------------
    model.eval()
    val_loss, val_err = 0, 0

    with torch.no_grad():
        for wav, targets, angle in val_loader:
            wav, targets, angle = wav.to(device), targets.to(device), angle.to(device)

            pred = model(wav)
            loss = criterion(pred, targets)

            val_loss += loss.item()
            val_err += angular_error_deg(pred, angle)

    avg_val_err = val_err / len(val_loader)
    avg_val_loss = val_loss / len(val_loader)

    ep_dur  = time.time() - ep_start
    ep_m, ep_s = divmod(int(ep_dur), 60)

    elapsed      = time.time() - total_start
    eta_sec      = (elapsed / epoch) * (NUM_EPOCHS - epoch)
    eta_m, eta_s = divmod(int(eta_sec), 60)

    cur_lr = optimizer.param_groups[0]['lr']

    history.append({
            'epoch': epoch,
            'train_loss': avg_train_loss,
            'train_err': avg_train_err,
            'val_loss': avg_val_loss,
            'val_err': avg_val_err,
            'lr': cur_lr
        })

    print(
        f"Epoch {epoch} | "
        f"LR: {cur_lr:.2e} | "
        f"Train → Loss: {avg_train_loss:.4f}  Hata: {avg_train_err:.1f}° | "
        f"Val → Loss: {avg_val_loss:.4f}  Hata: {avg_val_err:.1f}° | "
        f"ETA: {eta_m}m{eta_s:02d}s"
    )

    scheduler.step(avg_val_err)

    early_stopping(avg_val_err, model)
    if early_stopping.early_stop:
        print("\n[!] Early Stopping: eğitim durduruldu.")
        break